In [67]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
from keras.preprocessing.image import load_img
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, Rescaling, RandomTranslation
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import load_model
tf.compat.v1.enable_eager_execution

%matplotlib inline
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [68]:
def make_model(learning_rate=0.01, momentum=0.8):

    

    #########################################

    inputs = keras.Input(shape=(200, 200, 3))
    c_layer = keras.layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu')(inputs)
    pooling = keras.layers.MaxPooling2D(pool_size=(2,2))(c_layer)
    
    flatten = keras.layers.Flatten()(pooling)
    dense_64 = keras.layers.Dense(64, activation='relu')(flatten)
    
    outputs = keras.layers.Dense(1, activation='sigmoid')(dense_64)
    
    model = keras.Model(inputs, outputs)
    
    #########################################

    optimizer = keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

## Question 1
Since we have a binary classification problem, what is the best loss function for us?

- mean squared error
- binary crossentropy [X]
- categorical crossentropy
- cosine similarity

Note: since we specify an activation for the output layer, we don't need to set from_logits=True

In [69]:
input_size = 200

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    './data/train',
    image_size=(input_size, input_size),
    batch_size=32
)

train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, depth=10)))

Found 801 files belonging to 2 classes.


In [70]:
model = make_model()
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 198, 198, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 99, 99, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 313632)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │    20,072,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,073,473 (76.57 MB)

 Trainable params: 20,073,473 (76.57 MB)

 Non-trainable params: 0 (0.00 B)

## Question 2
What's the total number of parameters of the model? You can use the summary method for that.

- 896
- 11214912
- 15896912
- 20072512 [X]

In [71]:
def rescale_function(image):
    return image * (1. / 255.0)

train_gen = ImageDataGenerator(preprocessing_function=rescale_function)
val_gen = ImageDataGenerator(preprocessing_function=rescale_function)

train_ds = train_gen.flow_from_directory(
    './data/train',
    target_size=(input_size, input_size),
    batch_size=20,
    shuffle=False,
    class_mode='binary'
)

val_ds = val_gen.flow_from_directory(
    './data/test',
    target_size=(input_size, input_size),
    batch_size=20,
    shuffle=False,
    class_mode='binary'
)

print(train_ds.class_mode)
print(val_ds.class_mode)

Found 800 images belonging to 2 classes.
Found 201 images belonging to 2 classes.
binary
binary


In [72]:
model = make_model()
history = model.fit(train_ds, epochs=10, validation_data=val_ds)
model.save('full_model.keras')

c:\Users\Juan Pablo\Documents\Data Science Learning\zoomcamp\intro\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 178ms/step - accuracy: 0.3851 - loss: 2.6975 - val_accuracy: 0.5373 - val_loss: 0.6924
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 166ms/step - accuracy: 0.6150 - loss: 0.6919 - val_accuracy: 0.7264 - val_loss: 0.6872
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 168ms/step - accuracy: 0.6125 - loss: 0.7057 - val_accuracy: 0.4876 - val_loss: 0.6938
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 176ms/step - accuracy: 0.3730 - loss: 0.7039 - val_accuracy: 0.4876 - val_loss: 0.6963
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 171ms/step - accuracy: 0.4972 - loss: 0.6972 - val_accuracy: 0.6219 - val_loss: 0.6817
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 174ms/step - accuracy: 0.6864 - loss: 0.6703 - val_accuracy: 0.6020 - val_loss: 0.6711
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 172ms/step - accuracy: 0.7630 - loss: 1.6021 - val_accuracy: 0.5124 - val_loss: 0.6807
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 172ms/step - accuracy: 0.4464 - loss: 0.6963 - val_accuracy: 0.

In [73]:
hist = history.history

In [74]:
med = np.median(hist['accuracy'])
med

0.5406250059604645

## Question 3
What is the median of training accuracy for all the epochs for this model?

- 0.10
- 0.32
- 0.50 [X]
- 0.72

In [75]:
std = np.std(hist['loss'])
std

0.24524745056529865

## Question 4
What is the standard deviation of training loss for all the epochs for this model?

- 0.028
- 0.068
- 0.128 [X]
- 0.168

In [76]:
def rescale_function(image):
    return image * (1. / 255.0)

input_size = 200

train_gen = ImageDataGenerator(preprocessing_function=rescale_function,
                               rotation_range=50,
                               width_shift_range=0.1,
                               height_shift_range=0.1,
                               zoom_range=0.1,
                               horizontal_flip=True,
                               fill_mode='nearest'
                               )

val_gen = ImageDataGenerator(preprocessing_function=rescale_function)

train_ds = train_gen.flow_from_directory(
    './data/train',
    target_size=(input_size, input_size),
    batch_size=20,
    shuffle=False,
    class_mode='binary'
)

val_ds = val_gen.flow_from_directory(
    './data/test',
    target_size=(input_size, input_size),
    batch_size=20,
    shuffle=False,
    class_mode='binary'
)

Found 800 images belonging to 2 classes.
Found 201 images belonging to 2 classes.


In [77]:
model = load_model('full_model.keras')
history = model.fit(train_ds, epochs=10, validation_data=val_ds)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 272ms/step - accuracy: 0.5512 - loss: 0.6803 - val_accuracy: 0.6766 - val_loss: 0.6558
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 260ms/step - accuracy: 0.5508 - loss: 0.6893 - val_accuracy: 0.6667 - val_loss: 0.6555
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 252ms/step - accuracy: 0.5719 - loss: 0.6774 - val_accuracy: 0.6866 - val_loss: 0.6461
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 254ms/step - accuracy: 0.6095 - loss: 0.6642 - val_accuracy: 0.6816 - val_loss: 0.6248
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 253ms/step - accuracy: 0.7025 - loss: 0.6241 - val_accuracy: 0.4975 - val_loss: 0.7074
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 258ms/step - accuracy: 0.4474 - loss: 0.7305 - val_accuracy: 0.6617 - val_loss: 0.6386
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 251ms/step - accuracy: 0.6052 - loss: 0.6656 - val_accuracy: 0.6716 - val_loss: 0.6248
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 252ms/step - accuracy: 0.6028 - loss: 0.6504 - val_accu

In [78]:
hist2 = history.history

In [80]:
np.mean(hist2['val_loss'])

0.6356666386127472

## Question 5
Let's train our model for 10 more epochs using the same code as previously.

Note: make sure you don't re-create the model - we want to continue training the model we already started training.

What is the mean of test loss for all the epochs for the model trained with augmentations?

- 0.26
- 0.56 [X]
- 0.86
- 1.16

In [86]:
np.mean(hist2['val_accuracy'][5:])

0.6716417908668518

## Question 6
What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?

- 0.31
- 0.51
- 0.71 [X]
- 0.91